In [12]:
from zipapp import create_archive

from dotenv import load_dotenv
from langchain.agents.middleware import before_model, after_model, before_agent, after_agent, AgentMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

load_dotenv(override=True)

model = init_chat_model(model="deepseek:deepseek-chat")

# Decorator

In [9]:
from typing import Any
from langgraph.runtime import Runtime
from langchain.agents import AgentState, create_agent


@before_model
def before_model_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> before_model <----\n"
    return None

@after_model
def after_model_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> after_model <----\n"
    return None

@before_agent
def before_agent_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> before_agent <----\n"
    return None

@after_agent
def after_agent_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> after_agent <----\n"
    return None

In [10]:
agent = create_agent(
    model=model,
    middleware=[
        before_model_middleware,
        after_model_middleware,
        before_agent_middleware,
        after_agent_middleware
    ],
)

response = agent.invoke({
    "messages": [
        HumanMessage("Hello!")
    ]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Hello!----> before_agent <----
----> before_model <----

================================== Ai Message ==================================

Hello! How can I help you today?----> after_model <----
----> after_agent <----


# Class

In [13]:
class MyMiddleware(AgentMiddleware):
    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> before_model <----\n"
        return None

    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> after_model <----\n"
        return None

    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> before_agent <----\n"
        return None

    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> after_agent <----\n"
        return None

In [15]:
my_middleware = MyMiddleware()

agent = create_agent(
    model=model,
    middleware=[my_middleware],
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Hello!----> before_agent <----
----> before_model <----

================================== Ai Message ==================================

Hello! How can I help you today?----> after_model <----
----> after_agent <----
